# LLM-Assisted Transaction Reason-Code Classifier

**Business Problem:**  
In a high-velocity F&B transaction environment (400–600 orders/day), exception triage requires manually labeling each discrepancy with a standardized reason code. This notebook automates that classification step using an LLM, reducing manual review time and improving consistency.

**Reason Code Taxonomy:**
- `Weekend Batch Delay` — Settlement timing gap due to weekend/holiday batching
- `Gateway Timeout` — Payment gateway failure or connection drop
- `Duplicate Entry` — Same transaction appears more than once in settlement files
- `Refund Mismatch` — Refund processed but not reconciled in ledger
- `Fee Deduction Gap` — Platform fee discrepancy not captured in revenue report
- `Unclassified` — LLM cannot confidently assign a reason code

In [ ]:
# Step 1: Install dependencies
!pip install openai pandas -q

In [ ]:
# Step 2: Set your OpenAI API Key
# Get your key from: https://platform.openai.com/api-keys
# IMPORTANT: Never share this key or upload it to GitHub

import os
os.environ['OPENAI_API_KEY'] = 'sk-YOUR-API-KEY-HERE'  # Replace with your key

In [ ]:
# Step 3: Load the transaction data
import pandas as pd

df = pd.read_csv('sample_transactions.csv')
print(f'Loaded {len(df)} transactions')
df.head()

In [ ]:
# Step 4: Define the classifier function
from openai import OpenAI

client = OpenAI()

REASON_CODES = [
    'Weekend Batch Delay',
    'Gateway Timeout',
    'Duplicate Entry',
    'Refund Mismatch',
    'Fee Deduction Gap',
    'Unclassified'
]

SYSTEM_PROMPT = """You are a financial operations analyst specializing in payment reconciliation.
Your job is to classify transaction discrepancy descriptions into standardized reason codes.

Available reason codes:
- Weekend Batch Delay: Settlement timing gap due to weekend or holiday batch processing
- Gateway Timeout: Payment gateway failure, connection error, or timeout
- Duplicate Entry: Same transaction appears more than once in settlement files
- Refund Mismatch: Refund processed but not correctly reconciled in the ledger
- Fee Deduction Gap: Platform fee or deduction discrepancy not captured in revenue report
- Unclassified: Cannot confidently assign any of the above codes

Respond with ONLY the reason code, nothing else. No explanation, no punctuation."""

def classify_transaction(description):
    """Send a transaction description to the LLM and return a reason code."""
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',  # Cost-efficient model, ~$0.001 per 1K tokens
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this discrepancy: {description}"}
            ],
            max_tokens=20,
            temperature=0  # Temperature=0 for consistent, deterministic output
        )
        label = response.choices[0].message.content.strip()
        # Validate output is a known reason code
        if label not in REASON_CODES:
            return 'Unclassified'
        return label
    except Exception as e:
        print(f'API error: {e}')
        return 'Unclassified'

print('Classifier function defined.')

In [ ]:
# Step 5: Test with one transaction first
test_description = df['description'].iloc[0]
test_result = classify_transaction(test_description)

print(f'Description: {test_description}')
print(f'LLM Label:   {test_result}')
print(f'Manual Label: {df["manual_label"].iloc[0]}')

In [ ]:
# Step 6: Run classifier on all transactions
import time

print('Classifying all transactions...')
llm_labels = []

for i, row in df.iterrows():
    label = classify_transaction(row['description'])
    llm_labels.append(label)
    print(f"  [{i+1}/{len(df)}] {row['transaction_id']} → {label}")
    time.sleep(0.5)  # Rate limit buffer

df['llm_label'] = llm_labels
print('\nDone.')

In [ ]:
# Step 7: Evaluate accuracy vs manual labels
df['match'] = df['llm_label'] == df['manual_label']
accuracy = df['match'].mean()

print(f'Classification Accuracy: {accuracy:.1%}')
print(f'Correct: {df["match"].sum()} / {len(df)}')
print()

# Show mismatches for review
mismatches = df[~df['match']][['transaction_id', 'description', 'manual_label', 'llm_label']]
if len(mismatches) > 0:
    print('Mismatches (flagged for manual review):')
    print(mismatches.to_string(index=False))
else:
    print('No mismatches — 100% accuracy.')

In [ ]:
# Step 8: Generate QA exception report
# Matches = auto-approved, Mismatches = flagged for manual review

auto_approved = df[df['match']].copy()
flagged = df[~df['match']].copy()

auto_approved['review_status'] = 'Auto-Approved'
flagged['review_status'] = 'Flagged for Manual Review'

output = pd.concat([auto_approved, flagged]).sort_values('transaction_id')
output = output[['transaction_id', 'timestamp', 'amount', 'platform',
                  'description', 'manual_label', 'llm_label', 'match', 'review_status']]

output.to_csv('output_classified.csv', index=False)
print(f'QA Exception Report saved: output_classified.csv')
print(f'  Auto-Approved: {len(auto_approved)}')
print(f'  Flagged for Review: {len(flagged)}')
output.head(10)

In [ ]:
# Step 9: Reason code distribution summary
print('=== Reason Code Distribution ===')
print(df['llm_label'].value_counts().to_string())
print()
print('=== Auto-Approved vs Flagged ===')
print(df['review_status'].value_counts().to_string() if 'review_status' in df.columns else output['review_status'].value_counts().to_string())

## Results Summary

This notebook demonstrates an end-to-end LLM-assisted classification workflow:

1. **Input**: Raw transaction discrepancy descriptions
2. **Process**: LLM classifies each description into a standardized reason code
3. **Validation**: LLM output compared against manual labels to compute accuracy
4. **Output**: QA exception report with auto-approved and flagged transactions

**Business Impact**:  
Automates the manual reason-code labeling step in reconciliation triage, reducing review time for standard exception types while routing edge cases to human review.

**Next Steps**:  
- Expand taxonomy with additional reason codes from real exception logs
- Add confidence scoring to improve flagging logic
- Connect to live transaction feed for real-time classification